# Maia Lite 0.3.0 — Benchmark comparativo Base × LoRA SFT v2.2

**Colab L4/A100 | inferência somente | sem treinamento**

Avalia o checkpoint original **step 300000** e o melhor adaptador **LoRA v2.2 (step 1800)** sob prompts idênticos, com três estratégias de decodificação. Salva respostas, métricas e planilha para avaliação humana no Drive.

**Interpretação:** métricas automáticas de repetição e comprimento não medem veracidade ou coerência. A planilha de avaliação humana é essencial. As perguntas fixas são um *probe* qualitativo, não um benchmark estatisticamente representativo; confira eventual sobreposição com o corpus de treinamento antes de alegar generalização.

**Execução:** rodar células de cima para baixo em uma sessão nova. O teste não depende de `val_raw`, `VAL_FILE`, células anteriores ou GPU de treinamento. Não modifica os checkpoints.

In [ ]:
# 1 — Instalação e ambiente
import sys, subprocess, os, json, math, gc, time, random, hashlib, re
from pathlib import Path
from dataclasses import dataclass
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
try:
    from transformers import AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers'])
    from transformers import AutoTokenizer
from google.colab import drive
assert torch.cuda.is_available(), 'Selecione GPU L4/A100 em Ambiente de execução > Alterar tipo.'
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
# 2 — Caminhos e configurações (alterar apenas se seu Drive usa outra estrutura)
ROOT = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat')
BASE_CKPT = ROOT/'training_runs/maia_lite_0_3_0_a100_v3_experimental/latest.pt'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
ADAPTER = ROOT/'training_runs/maia_lite_0_3_0_sft_v2_2_lora_v1/adapter_best.pt'
OUTPUT_DIR = ROOT/'evaluations/maia_lite_0_3_0_base_vs_lora_v2_2'
for p in (BASE_CKPT, ADAPTER):
    assert p.is_file(), f'Arquivo não encontrado: {p}'
assert TOKENIZER_DIR.is_dir(), f'Tokenizer não encontrado: {TOKENIZER_DIR}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Execução padrão: 9 prompts x 2 modelos x 3 decodificações = 54 respostas.
# Para teste rápido, defina QUICK_MODE=True (3 prompts x 2 x 3 = 18 respostas).
QUICK_MODE = False
MAX_NEW_TOKENS = 96
SEED = 20261118
TEMPERATURE = 0.7
TOP_K = 40
TOP_P = 0.9
REPETITION_PENALTY = 1.18
assert MAX_NEW_TOKENS > 0
print('Base:', BASE_CKPT)
print('LoRA:', ADAPTER)
print('Resultados:', OUTPUT_DIR)

## Arquitetura idêntica ao treinador original

A arquitetura MaiaGPT, os nomes dos módulos (`qkv`, `proj`), a implementação LoRA e o vínculo dos embeddings são reproduzidos do notebook de SFT v2.2 enviado. O modelo-base é carregado uma vez; a comparação desativa/ativa matematicamente a contribuição LoRA sem alterar os pesos originais.

In [ ]:
# 3 — Arquitetura original
@dataclass
class Config:
    vocab_size: int = 50257
    block_size: int = 2048
    n_layer: int = 36
    n_head: int = 16
    n_embd: int = 1024
    dropout: float = 0.0
    bias: bool = True
cfg = Config()
class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d=c.n_embd
        self.ln1=nn.LayerNorm(d)
        self.qkv=nn.Linear(d,3*d,bias=c.bias)
        self.proj=nn.Linear(d,d,bias=c.bias)
        self.ln2=nn.LayerNorm(d)
        self.fc=nn.Linear(d,4*d,bias=c.bias)
        self.out=nn.Linear(4*d,d,bias=c.bias)
        self.nh=c.n_head
    def forward(self,x):
        b,t,d=x.shape
        q,k,v=self.qkv(self.ln1(x)).chunk(3,dim=-1)
        q=q.view(b,t,self.nh,d//self.nh).transpose(1,2)
        k=k.view(b,t,self.nh,d//self.nh).transpose(1,2)
        v=v.view(b,t,self.nh,d//self.nh).transpose(1,2)
        a=F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=0.0)
        x=x+self.proj(a.transpose(1,2).contiguous().view(b,t,d))
        x=x+self.out(F.gelu(self.fc(self.ln2(x)),approximate='tanh'))
        return x
class MaiaGPT(nn.Module):
    def __init__(self,c):
        super().__init__(); self.cfg=c
        self.tok=nn.Embedding(c.vocab_size,c.n_embd)
        self.pos=nn.Embedding(c.block_size,c.n_embd)
        self.blocks=nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm=nn.LayerNorm(c.n_embd)
        self.head=nn.Linear(c.n_embd,c.vocab_size,bias=False)
        self.head.weight=self.tok.weight
    def forward(self,idx,targets=None):
        b,t=idx.shape
        assert t<=self.cfg.block_size
        x=self.tok(idx)+self.pos(torch.arange(t,device=idx.device))
        for block in self.blocks: x=block(x)
        x=self.norm(x)
        logits=self.head(x)
        loss=None if targets is None else F.cross_entropy(logits.reshape(-1,logits.size(-1)).float(),targets.reshape(-1),ignore_index=-100)
        return logits,loss
class LoRALinear(nn.Module):
    def __init__(self,base,rank,alpha,dropout):
        super().__init__()
        assert isinstance(base,nn.Linear)
        self.base=base
        self.scale=alpha/rank
        self.active=True
        self.dropout=nn.Dropout(dropout)
        self.A=nn.Parameter(torch.empty(rank,base.in_features,device=base.weight.device,dtype=torch.float32))
        self.B=nn.Parameter(torch.zeros(base.out_features,rank,device=base.weight.device,dtype=torch.float32))
        nn.init.kaiming_uniform_(self.A,a=math.sqrt(5))
    def forward(self,x):
        if not self.active:
            return self.base(x)
        update=F.linear(F.linear(self.dropout(x).float(),self.A),self.B)*self.scale
        return self.base(x)+update.to(x.dtype)


In [ ]:
# 4 — Carregamento do checkpoint-base e do adaptador (somente leitura)
# weights_only=False SOMENTE para artefatos produzidos pelo próprio projeto, que são confiáveis.
tokenizer=AutoTokenizer.from_pretrained(str(TOKENIZER_DIR),local_files_only=True,use_fast=True)
assert len(tokenizer)==cfg.vocab_size, (len(tokenizer),cfg.vocab_size)
assert tokenizer.eos_token_id is not None
print('Carregando base...')
ckpt=torch.load(BASE_CKPT,map_location='cpu',weights_only=False)
assert int(ckpt['step'])==300000, f"Esperado step 300000, recebido {ckpt['step']}"
model=MaiaGPT(cfg)
model.load_state_dict(ckpt['model'],strict=True)
del ckpt; gc.collect()
assert model.head.weight.data_ptr()==model.tok.weight.data_ptr()
model=model.to(device='cuda',dtype=torch.bfloat16)
for p in model.parameters(): p.requires_grad_(False)
print('Carregando LoRA...')
saved=torch.load(ADAPTER,map_location='cpu',weights_only=False)
assert 'adapter' in saved and 'config' in saved, 'Formato do adaptador inesperado'
config=saved['config']
assert config.get('base_step')==300000, 'Adaptador não corresponde ao checkpoint-base esperado'
assert config.get('target_modules')==['qkv','proj'], 'Módulos LoRA inesperados'
for block in model.blocks:
    block.qkv=LoRALinear(block.qkv,int(config['rank']),float(config['alpha']),float(config['dropout']))
    block.proj=LoRALinear(block.proj,int(config['rank']),float(config['alpha']),float(config['dropout']))
lora_modules=[m for m in model.modules() if isinstance(m,LoRALinear)]
expected={k for k in model.state_dict() if k.endswith('.A') or k.endswith('.B')}
assert set(saved['adapter'])==expected, 'Chaves LoRA incompatíveis com o modelo'
with torch.no_grad():
    for name,p in model.named_parameters():
        if name in saved['adapter']:
            p.copy_(saved['adapter'][name].to(device=p.device,dtype=p.dtype))
ADAPTER_STEP=int(saved.get('update',-1))
ADAPTER_VAL_LOSS=float(saved.get('val_loss',float('nan')))
del saved; gc.collect()
model.eval()
def set_lora(enabled):
    for m in lora_modules: m.active=bool(enabled)
print('Modelo carregado; LoRA step:',ADAPTER_STEP,'; loss:',ADAPTER_VAL_LOSS)
print('Módulos LoRA:',len(lora_modules))
assert len(lora_modules)==cfg.n_layer*2


In [ ]:
# 5 — Teste de integridade: desligar/ligar LoRA altera os logits?
probe=tokenizer('Qual é a importância das árvores?',return_tensors='pt',add_special_tokens=False)['input_ids'].to('cuda')
with torch.inference_mode(), torch.autocast('cuda',dtype=torch.bfloat16):
    set_lora(False); base_logits,_=model(probe)
    set_lora(True); tuned_logits,_=model(probe)
diff=(base_logits[:,-1,:].float()-tuned_logits[:,-1,:].float()).abs()
print('Diferença máxima dos logits:',float(diff.max()))
assert torch.isfinite(diff).all(), 'Logits inválidos'
assert float(diff.max())>0, 'LoRA parece não alterar o modelo'
del base_logits,tuned_logits,diff


## Perguntas fixas, três idiomas

Nove prompts de domínio geral, com dificuldades distintas e gabaritos conceituais curtos para orientar a revisão humana. **Não** são exemplos retirados automaticamente da validação. A formulação segue as instruções usadas no SFT; isso não garante que o conjunto seja livre de sobreposição com o treino.

In [ ]:
# 6 — Benchmark de perguntas (editável)
INSTRUCTIONS={
 'pt':'Responda à pergunta de forma clara, objetiva e cientificamente correta. Se não souber, indique a incerteza. ',
 'en':'Answer clearly, concisely, and accurately. If unsure, acknowledge uncertainty. ',
 'es':'Responde de forma clara, concisa y precisa. Si no lo sabes, indica la incertidumbre. '
}
QUESTIONS=[
 {'id':'pt_trees','language':'pt','question':'Quais são os principais benefícios do plantio de árvores nas cidades?','reference':'Sombra, redução de calor, qualidade do ar, biodiversidade e manejo de águas pluviais.'},
 {'id':'pt_photosynthesis','language':'pt','question':'O que é fotossíntese e qual é sua importância?','reference':'Conversão de energia luminosa em energia química; fixação de carbono e produção de matéria orgânica.'},
 {'id':'pt_uncertainty','language':'pt','question':'Sem acesso ao artigo, quais são as duas técnicas exatas de decomposição celular mencionadas nele?','reference':'Não é possível identificar as técnicas exatas sem o texto do artigo.'},
 {'id':'en_kanji','language':'en','question':'How is the use of Chinese characters in Japanese different from their use in Chinese languages?','reference':'Japanese uses kanji alongside kana, with Japanese-specific readings; Chinese uses hanzi as its principal script.'},
 {'id':'en_gravity','language':'en','question':'Why do objects fall toward Earth when dropped?','reference':'Earth’s gravity accelerates objects toward its center.'},
 {'id':'en_uncertainty','language':'en','question':'Without seeing a specific research paper, what exact numerical result did its authors report?','reference':'Cannot know the exact result without the paper.'},
 {'id':'es_trees','language':'es','question':'¿Cuáles son las ventajas de plantar árboles para la población?','reference':'Sombra, menor temperatura urbana, biodiversidad, calidad del aire y bienestar.'},
 {'id':'es_water','language':'es','question':'¿Por qué es importante el agua para los seres vivos?','reference':'Participa en reacciones bioquímicas, transporte de sustancias y regulación térmica.'},
 {'id':'es_uncertainty','language':'es','question':'Sin consultar el artículo, ¿cuál fue el resultado numérico exacto de su experimento?','reference':'No se puede saber el resultado exacto sin consultar el artículo.'}
]
if QUICK_MODE:
    QUESTIONS=[next(q for q in QUESTIONS if q['id']==x) for x in ['pt_trees','en_kanji','es_trees']]
for q in QUESTIONS:
    q['prompt']=INSTRUCTIONS[q['language']]+q['question']
print('Perguntas:',len(QUESTIONS))


In [ ]:
# 7 — Decodificação: greedy, top-k/top-p, top-k/top-p + repetition penalty
# Seeds fixas por prompt/estratégia; modelos comparados com o mesmo seed.
STRATEGIES={
 'greedy': {'sample':False,'repetition_penalty':1.0},
 'topk_topp': {'sample':True,'repetition_penalty':1.0},
 'topk_topp_rep': {'sample':True,'repetition_penalty':REPETITION_PENALTY},
}
def apply_repetition_penalty(logits, token_ids, penalty):
    if penalty==1.0: return logits
    # Penaliza tokens já presentes na sequência gerada; não penaliza o prompt.
    if token_ids:
        used=torch.tensor(sorted(set(token_ids)),device=logits.device,dtype=torch.long)
        vals=logits[used]
        logits[used]=torch.where(vals<0,vals*penalty,vals/penalty)
    return logits

def choose_token(logits,strategy,generator):
    if not strategy['sample']:
        return int(torch.argmax(logits).item())
    logits=logits/TEMPERATURE
    if TOP_K is not None and TOP_K>0:
        cutoff=torch.topk(logits,min(TOP_K,logits.numel())).values[-1]
        logits=logits.masked_fill(logits<cutoff,-float('inf'))
    if TOP_P<1.0:
        sorted_logits,indices=torch.sort(logits,descending=True)
        sorted_probs=torch.softmax(sorted_logits,dim=-1)
        cumulative=torch.cumsum(sorted_probs,dim=-1)
        remove=cumulative-sorted_probs>=TOP_P
        sorted_logits[remove]=-float('inf')
        probs=torch.softmax(sorted_logits,dim=-1)
        ix=torch.multinomial(probs,1,generator=generator)
        return int(indices[ix].item())
    probs=torch.softmax(logits,dim=-1)
    return int(torch.multinomial(probs,1,generator=generator).item())

@torch.inference_mode()
def generate(prompt,strategy_name,seed,max_new_tokens=MAX_NEW_TOKENS):
    strategy=STRATEGIES[strategy_name]
    input_ids=tokenizer(prompt,add_special_tokens=False,return_tensors='pt')['input_ids'].to('cuda')
    assert input_ids.shape[1]+max_new_tokens<=cfg.block_size, 'Prompt excede contexto disponível'
    generator=torch.Generator(device='cuda');generator.manual_seed(seed)
    produced=[]
    stop_reason='max_tokens'
    for _ in range(max_new_tokens):
        with torch.autocast('cuda',dtype=torch.bfloat16):
            logits,_=model(input_ids)
        last=logits[0,-1,:].float().clone()
        last=apply_repetition_penalty(last,produced,strategy['repetition_penalty'])
        token_id=choose_token(last,strategy,generator)
        if token_id==tokenizer.eos_token_id:
            stop_reason='eos';break
        produced.append(token_id)
        input_ids=torch.cat([input_ids,torch.tensor([[token_id]],device='cuda')],dim=1)
    return {'response':tokenizer.decode(produced,skip_special_tokens=True).strip(),
            'tokens_generated':len(produced),'stop_reason':stop_reason}


In [ ]:
# 8 — Métricas automáticas exploratórias (não substituem julgamento humano)
def repetition_metrics(text):
    words=re.findall(r'\w+',text.lower(),flags=re.UNICODE)
    if not words: return {'repeated_3gram_fraction':0.0,'distinct_2':0.0,'distinct_3':0.0,'longest_repeated_run':0}
    def distinct(n):
        grams=list(zip(*(words[i:] for i in range(n))))
        return len(set(grams))/len(grams) if grams else 0.0
    grams=list(zip(*(words[i:] for i in range(3))))
    counts=Counter(grams)
    repeated=sum(c-1 for c in counts.values() if c>1)
    # maior sequência contígua de palavras repetida imediatamente
    longest=0
    for width in range(1,min(20,len(words)//2)+1):
        for i in range(len(words)-2*width+1):
            if words[i:i+width]==words[i+width:i+2*width]: longest=max(longest,width)
    return {'repeated_3gram_fraction':repeated/max(1,len(grams)),
            'distinct_2':distinct(2),'distinct_3':distinct(3),
            'longest_repeated_run':longest}

# Identidade estável das perguntas para comparação entre execuções
BENCHMARK_HASH=hashlib.sha256(json.dumps(QUESTIONS,ensure_ascii=False,sort_keys=True).encode()).hexdigest()
print('SHA256 das perguntas:',BENCHMARK_HASH)


In [ ]:
# 9 — Executar comparação pareada e salvar resultados incrementalmente
# Caso interrompa, as respostas já concluídas permanecem em results.jsonl.
RESULTS_FILE=OUTPUT_DIR/'results.jsonl'
RUN_CONFIG={'base_checkpoint':str(BASE_CKPT),'adapter':str(ADAPTER),'adapter_step':ADAPTER_STEP,
 'adapter_val_loss':ADAPTER_VAL_LOSS,'benchmark_sha256':BENCHMARK_HASH,
 'seed':SEED,'max_new_tokens':MAX_NEW_TOKENS,'temperature':TEMPERATURE,
 'top_k':TOP_K,'top_p':TOP_P,'repetition_penalty':REPETITION_PENALTY,
 'quick_mode':QUICK_MODE,'strategies':STRATEGIES}
RUN_ID=hashlib.sha256(json.dumps(RUN_CONFIG,sort_keys=True).encode()).hexdigest()[:16]
(OUTPUT_DIR/'run_config.json').write_text(json.dumps({**RUN_CONFIG,'run_id':RUN_ID},indent=2,ensure_ascii=False),encoding='utf-8')
rows=[]
if RESULTS_FILE.exists():
    with RESULTS_FILE.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                try:
                    row=json.loads(line)
                    if row.get('run_id')==RUN_ID: rows.append(row)
                except json.JSONDecodeError: pass
completed={(r['model'],r['strategy'],r['id']) for r in rows}
print('Execuções existentes nesta configuração:',len(completed))
for q in QUESTIONS:
    for strategy in STRATEGIES:
        # Seed igual nos dois modelos para a mesma pergunta e estratégia
        key=f"{SEED}|{q['id']}|{strategy}"
        seed=int(hashlib.sha256(key.encode()).hexdigest()[:8],16)
        for model_name,enabled in [('base',False),('lora_v2_2',True)]:
            if (model_name,strategy,q['id']) in completed: continue
            set_lora(enabled)
            t0=time.time()
            result=generate(q['prompt'],strategy,seed)
            row={'run_id':RUN_ID,'id':q['id'],'language':q['language'],'question':q['question'],
                 'prompt':q['prompt'],'reference':q['reference'],'model':model_name,
                 'strategy':strategy,'seed':seed,'elapsed_s':round(time.time()-t0,2),
                 **result,**repetition_metrics(result['response'])}
            rows.append(row)
            with RESULTS_FILE.open('a',encoding='utf-8') as f:
                f.write(json.dumps(row,ensure_ascii=False)+'\n')
            print(f"{len(rows):>3} | {q['id']:<19} | {strategy:<14} | {model_name:<9} | {result['tokens_generated']:>3} tokens | {row['elapsed_s']:>5.1f}s",flush=True)
set_lora(True)
print('Finalizado. Registros nesta execução:',len(rows))


In [ ]:
# 10 — Tabelas, resumos e exportação
results_df=pd.DataFrame(rows)
assert not results_df.empty
results_df=results_df.sort_values(['language','id','strategy','model']).reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR/'generations.csv',index=False,encoding='utf-8-sig')
summary=(results_df.groupby(['model','strategy','language'],as_index=False)
         .agg(n=('id','count'),mean_tokens=('tokens_generated','mean'),
              mean_repeated_3gram_fraction=('repeated_3gram_fraction','mean'),
              mean_distinct_3=('distinct_3','mean'),
              mean_seconds=('elapsed_s','mean')))
summary.to_csv(OUTPUT_DIR/'automatic_summary.csv',index=False,encoding='utf-8-sig')
print('RESUMO AUTOMÁTICO (não é nota de qualidade factual):')
display(summary.round(3))
print('\nRESPOSTAS PAREADAS:')
pairs=results_df.pivot_table(index=['language','id','question','strategy'],columns='model',values='response',aggfunc='first').reset_index()
display(pairs)
pairs.to_csv(OUTPUT_DIR/'paired_responses.csv',index=False,encoding='utf-8-sig')


In [ ]:
# 11 — Ficha para avaliação humana, cega quanto ao modelo (opcional)
# Avalie sem consultar 'model' na tabela original. 0=ruim, 1=parcial, 2=bom.
# Coerência, relevância, precisão e seguimento de instruções requerem julgamento humano.
review=results_df[['run_id','id','language','question','reference','strategy','model','response']].copy()
review['blind_id']=review.apply(lambda r:hashlib.sha256((r['run_id']+'|'+r['id']+'|'+r['strategy']+'|'+r['model']).encode()).hexdigest()[:12],axis=1)
key=review[['blind_id','model']].copy()
key.to_csv(OUTPUT_DIR/'blind_key.csv',index=False,encoding='utf-8-sig')
review=review.drop(columns=['model'])
review=review.sample(frac=1,random_state=SEED).reset_index(drop=True)
for col in ['relevance_0_2','coherence_0_2','factuality_0_2','instruction_following_0_2','uncertainty_handling_0_2','reviewer_notes']:
    review[col]=''
review.to_csv(OUTPUT_DIR/'human_review_blind.csv',index=False,encoding='utf-8-sig')
print('Arquivos gerados em:',OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file(): print('-',p.name)
print('\nInterpretação: compare as notas humanas pareadas; não conclua superioridade com base apenas nas métricas automáticas.')


## Como interpretar

1. Compare **base × LoRA** para a mesma pergunta e decodificação. A ativação LoRA é a única diferença entre os modelos.
2. Verifique se a penalização reduz repetições **sem** deteriorar o conteúdo.
3. Respostas que alegam resultados específicos de artigos ausentes devem reconhecer incerteza.
4. Não use a menor loss do SFT como evidência isolada de melhoria conversacional.
5. Para conclusão científica robusta, amplie o benchmark com perguntas externas, deduplicação contra treino e avaliações humanas cegas por mais de um avaliador.

**Saídas:** `results.jsonl`, `generations.csv`, `paired_responses.csv`, `automatic_summary.csv`, `human_review_blind.csv`, `blind_key.csv`, `run_config.json`. O notebook **não** treina, sobrescreve ou altera checkpoints.